Traditional clustering methods, such as **K-means**, assume that every observation belongs to exactly one cluster. However, many real-world datasets contain observations that naturally share characteristics with multiple groups. For example, a music listener may enjoy both classical and jazz, or a customer may exhibit purchasing behaviors similar to several market segments. In these situations, assigning each observation to a single cluster can oversimplify the underlying patterns in the data.

**Gaussian Mixture Models (GMMs)** provide a probabilistic approach to clustering that addresses this limitation. Instead of assigning observations to a single cluster, GMMs estimate the **probability** that each observation belongs to every cluster. In this lesson, you'll explore how GMMs differ from hard clustering methods, understand the role of the **Expectation-Maximization (EM)** algorithm, and apply GMMs to real datasets using scikit-learn. You'll also investigate how model parameters influence clustering results and learn when probabilistic clustering is the most appropriate choice.

By the end of this lesson, you should be able to:

1. Explain the concept of soft (probabilistic) clustering and determine when it is more appropriate than hard clustering methods
2. Describe how Gaussian Mixture Models (GMMs) use Gaussian distributions and the Expectation-Maximization (EM) algorithm to estimate cluster membership probabilities
3. Compare Gaussian Mixture Models with K-means and hierarchical clustering, highlighting the assumptions, outputs, and strengths of each approach
4. Implement Gaussian Mixture Models using scikit-learn and interpret the resulting probabilistic cluster assignments 
5. Evaluate the effect of changing the number of clusters and covariance types on GMM performance, and explain how these parameters influence the resulting model. 

---

1. **Soft vs. Hard Cluster Assignment**

- **The Key Difference:** A hard cluster assignment forces each data point to belong exclusively to one single cluster (like in K-Means), whereas a soft cluster assignment assigns each point a set of probabilities (responsibilities) indicating how likely it is to belong to every cluster.
- **Why It Matters in the Real World:** Real-world data rarely has crisp boundaries, and data points often sit in overlapping zones between groups. Soft assignments capture uncertainty and mixed membership—for example, in customer segmentation, a shopper might exhibit 60% budget-buyer behavior and 40% luxury-buyer behavior, or in medical diagnostics, a patient's symptoms might overlap between two conditions. Knowing the probability lets decision-makers treat borderline cases differently from clear-cut ones.

2. **How Expectation-Maximization (EM) Fits a GMM**

- **What EM Does:** The EM algorithm finds the parameters (means, covariances, and mixing weights) of the Gaussian components by alternating between two steps:
	1. **E-Step (Expectation):** Using the current guesses for the Gaussian curves, it calculates the probability ("responsibility") that each data point was generated by each cluster.
	2. **M-Step (Maximization):** Using those probabilities as weights, it updates the center (mean), spread/shape (covariance), and size (mixing weight) of each Gaussian curve to better fit the data points assigned to it.
- **Why It Must Repeat:** There is a "chicken-and-egg" dependency: you cannot know which cluster a point belongs to without knowing where the Gaussian curves are, and you cannot accurately compute the Gaussian curves without knowing which points belong to them. Repeating the E and M steps iteratively refines both estimates until the parameters stabilize and converge on a local optimum.

3. **Spherical vs. Full Covariance Types**

- **How the Clustering Results Change:** Switching from spherical to full covariance allows clusters to stretch, tilt, and adapt to diagonal or elongated patterns in the data rather than splitting a single elongated group into multiple round pieces.
- **What It Reveals About Cluster Shapes:**
	- Spherical covariance constrains every cluster to have equal variance in all directions with zero correlation between features, meaning it can only represent circular (in 2D) or hyperspherical (in higher dimensions) clusters.
	- Full covariance lets each cluster learn its own independent variance along every axis as well as correlations between features, enabling it to represent elliptical clusters of any size, elongation, and orientation (tilt).

Unlike K-Means, which assigns observations based on distance to cluster centroids, GMM uses probability distributions to assign observations to clusters. It supports soft clustering, meaning each observation can have a probability of belonging to every cluster.

We will use the Iris dataset to:
1. Load and explore the dataset.
2. Standardise the features.
3. Determine a suitable number of components using AIC and BIC.
4. Train a GMM model.
5. Predict cluster assignments and probabilities.
6. Visualise the clusters.
7. Evaluate and interpret the results using the actual Iris species labels.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

In [3]:
# Load dataset
iris = load_iris()

# Create DataFrame
df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

# Add actual species labels
df['species'] = pd.Categorical.from_codes(
    iris.target,
    iris.target_names
)

In [4]:
# Display first five rows
display(df.head())



,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:
# Dataset information
print(df.info())



<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   sepal length (cm)  150 non-null    float64 
 1   sepal width (cm)   150 non-null    float64 
 2   petal length (cm)  150 non-null    float64 
 3   petal width (cm)   150 non-null    float64 
 4   species            150 non-null    category
dtypes: category(1), float64(4)
memory usage: 5.0 KB
None


In [6]:
# Check missing values
print(df.isnull().sum())


sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
species              0
dtype: int64


In [7]:
# Check species distribution
print(df['species'].value_counts())

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


The dataset contains 150 observations, with 50 observations for each species. The four features are:
1. Sepal length
2. Sepal width
3. Petal length
4. Petal width

We will use only the four numerical features to train the GMM. The actual species labels will be reserved for evaluating the clustering results.

In [8]:
# Create the features matrix: Drop the species column
X = df.drop('species', axis=1)


In [9]:
# Display descriptive statistics of the features
display(X.describe().T)

,count,mean,std,min,25%,50%,75%,max
sepal length (cm),150.0,5.843333,0.828066,4.3,5.1,5.80,6.4,7.9
sepal width (cm),150.0,3.057333,0.435866,2.0,2.8,3.00,3.3,4.4
petal length (cm),150.0,3.758000,1.765298,1.0,1.6,4.35,5.1,6.9
petal width (cm),150.0,1.199333,0.762238,0.1,0.3,1.30,1.8,2.5


#### **Standardise the features**
Standardisation is important because the four features have different measurement ranges.

In [10]:
# Standardise the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

This transforms each feature to approximately zero mean and unit variance.

--- 
#### **Determine the optimal number of components using AIC and BIC**
GMM requires us to specify the number of Gaussian components.

We can compare different component counts using:
- AIC (Akaike Information Criterion): Balances model fit against model complexity.
- BIC (Bayesian Information Criterion): Also penalises model complexity, generally more strongly than AIC.
For both metrics, lower values indicate a preferable model among those compared.